# Chapter 1 notebook — PyTorch inference & basic latency

This notebook walks through the Chapter 3 doc (`docs/03_model_inference_basics.md`) interactively:

1. Load a pretrained classifier in inference mode.
2. Compare *wrong* and *right* ways to time it.
3. Show cold vs warm latency.
4. Compare batch sizes.
5. Plot a latency histogram and print P50 / P95.

Run on the laptop CPU first. If you have CUDA, the last section will also run on GPU.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import models

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

## 1. Load a pretrained classifier

MobileNetV3-Small is a small ImageNet classifier (~2.5M params) that runs well on a laptop CPU. The weights bundle the preprocessing transform that matches how the model was trained.

In [ ]:
weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = models.mobilenet_v3_small(weights=weights)
preprocess = weights.transforms()
classes = weights.meta['categories']

model.eval().to(DEVICE)

param_count = sum(p.numel() for p in model.parameters())
print(f'model params: {param_count / 1e6:.2f} M')
print(f'in training mode? {model.training}')

## 2. Prepare an input image

We use the smoke-test image at `datasets/sample.jpg`. The exact prediction is unimportant — the goal is timing.

In [ ]:
img_path = Path('../datasets/sample.jpg')
assert img_path.exists(), f'missing {img_path}; run the repo from its root or adjust the path'

img = Image.open(img_path).convert('RGB')
x = preprocess(img).unsqueeze(0).to(DEVICE)
print('input tensor:', x.shape, x.dtype)

## 3. Wrong vs right inference setup

Compare four configurations:

| Setup | `.eval()` | `torch.no_grad()` | Warm-up |
|---|---|---|---|
| (A) Wrong: training mode + grads on | no | no | no |
| (B) Eval mode, but grads still on | yes | no | no |
| (C) Right: eval + no_grad, no warm-up | yes | yes | no |
| (D) Right: eval + no_grad + warm-up | yes | yes | yes |

We time 50 forward passes per configuration.

In [ ]:
def time_loop(fn, n=50):
    times = []
    for _ in range(n):
        start = time.perf_counter()
        fn()
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000.0)
    return np.array(times)

# (A) Training mode, grads on
model.train()
def fn_a():
    _ = model(x)
t_a = time_loop(fn_a)

# (B) Eval, but grads still tracked
model.eval()
def fn_b():
    _ = model(x)
t_b = time_loop(fn_b)

# (C) Eval + no_grad, no warm-up
model.eval()
def fn_c():
    with torch.no_grad():
        _ = model(x)
t_c = time_loop(fn_c)

# (D) Eval + no_grad + warm-up
model.eval()
for _ in range(10):
    with torch.no_grad():
        _ = model(x)
t_d = time_loop(fn_c)  # same fn as C, just after warm-up

for name, t in [('A train+grad', t_a), ('B eval+grad', t_b),
                ('C eval+nograd cold', t_c), ('D eval+nograd warm', t_d)]:
    print(f'{name:25s}  mean={t.mean():6.2f} ms  P50={np.percentile(t, 50):6.2f}  P95={np.percentile(t, 95):6.2f}')

You should see (A) and (B) significantly slower than (C) and (D) because of autograd overhead and BN/dropout differences. (D) — eval + no_grad + warm-up — is the deployment configuration. Use it everywhere.

## 4. Cold vs warm: the first iterations

Without warm-up, the first few iterations are dominated by one-time setup costs.

In [ ]:
# Re-create the model from scratch to truly start cold
model_fresh = models.mobilenet_v3_small(weights=weights).eval().to(DEVICE)

first_n = []
for _ in range(30):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model_fresh(x)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    first_n.append((time.perf_counter() - start) * 1000.0)

plt.figure(figsize=(9, 3.5))
plt.plot(range(1, len(first_n) + 1), first_n, marker='o', linewidth=1)
plt.axhline(np.median(first_n[5:]), color='gray', linestyle='--', label='median of iters 6+ (steady)')
plt.xlabel('iteration')
plt.ylabel('latency (ms)')
plt.title('Cold-start: first 30 inference calls')
plt.legend()
plt.tight_layout()
plt.show()

print(f'first call: {first_n[0]:.2f} ms')
print(f'iter 5+ median: {np.median(first_n[5:]):.2f} ms')

Typically you will see the first call is several times slower than the steady state. **Always discard the warm-up iterations when reporting latency.**

## 5. Single image vs small batch

Larger batches amortize fixed costs. *But* if you deploy at batch=1, only the batch=1 number matters.

In [ ]:
model.eval()
for _ in range(10):
    with torch.no_grad():
        _ = model(x)  # warm-up

for B in (1, 4, 8, 16, 32):
    xb = x.repeat(B, 1, 1, 1)
    times = []
    for _ in range(30):
        start = time.perf_counter()
        with torch.no_grad():
            _ = model(xb)
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000.0)
    times = np.array(times)
    per_image = times / B
    print(f'batch={B:>2d}  total mean={times.mean():6.2f} ms  per-image={per_image.mean():6.2f} ms')

Per-image latency typically drops as batch grows — but real-time camera systems run at batch=1, so the *batch=1* row is the deployment-relevant one.

## 6. Latency histogram and percentiles

Single-number averages hide the tail. Always report at least P50 and P95.

In [ ]:
n_iters = 200
model.eval()
for _ in range(10):
    with torch.no_grad():
        _ = model(x)  # warm-up

times = []
for _ in range(n_iters):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(x)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    times.append((time.perf_counter() - start) * 1000.0)
times = np.array(times)

print(f'iters={n_iters}  mean={times.mean():.2f} ms')
for p in (50, 90, 95, 99):
    print(f'  P{p:<2d} = {np.percentile(times, p):.2f} ms')

plt.figure(figsize=(9, 3.5))
plt.hist(times, bins=30, edgecolor='black', alpha=0.8)
for p, color in [(50, 'tab:blue'), (95, 'tab:red'), (99, 'tab:purple')]:
    plt.axvline(np.percentile(times, p), color=color, linestyle='--',
                label=f'P{p}={np.percentile(times, p):.2f} ms')
plt.xlabel('latency (ms)')
plt.ylabel('count')
plt.title(f'Latency distribution ({n_iters} steady-state iters, device={DEVICE.type})')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Top-5 prediction

Sanity-check the model output on the sample image.

In [ ]:
with torch.no_grad():
    probs = F.softmax(model(x), dim=1)
topp, topi = probs.topk(5, dim=1)
for p, i in zip(topp.squeeze(0).tolist(), topi.squeeze(0).tolist()):
    print(f'  {p * 100:5.2f}%  {classes[i]}')


## 8. Take-aways

1. Always `model.eval()` + `torch.no_grad()` for inference. The combination is non-negotiable.
2. Always warm up before timing. Discard the first ~10 iterations.
3. Always call `torch.cuda.synchronize()` on CUDA before stopping the timer.
4. Always report at least mean + P50 + P95 — never just mean.
5. Benchmark in the configuration you deploy (batch=1 for real-time).

Next: **Chapter 4** generalizes this into a reusable benchmark script that also reports memory and FPS, and **Chapter 5** moves the model to ONNX so we can compare runtimes side by side.